# Thu thập chữ cái VSL (quay 1 lần + tự thêm nhiễu)

Chữ cái trong VSL thường là tư thế tay **đứng yên**, nên mỗi lớp chỉ cần quay **1 mẫu**. Sau đó notebook tự tạo thêm bản nhiễu (`src.data.augment.augment_sample`) cho đến khi lớp có thêm đủ **số mẫu** đã nhập.

| Phím | Tác dụng |
|---|---|
| `SPACE` | bắt đầu (đếm ngược → quay `RECORD_SECONDS` giây) |
| `ENTER` | lưu mẫu vừa quay (chỉ khi chất lượng đạt) |
| `F` | lưu dù chất lượng thấp |
| `R` | bỏ mẫu vừa quay |
| `U` | xoá mẫu vừa lưu gần nhất (khi đang ở trạng thái sẵn sàng) |
| `N` | sang lớp kế tiếp |
| `Q` | thoát |

Chạy bằng Jupyter **cục bộ** trên máy có webcam.
Các bản nhiễu ghi `origin_sample_id` của mẫu gốc nên luôn nằm cùng fold với mẫu gốc, và `train.ipynb` chỉ chấm điểm trên mẫu gốc. Vì vậy mỗi chữ cái chỉ có **1 mẫu để kiểm định**: hãy quay thêm ở các buổi khác hoặc với người ký khác để đánh giá đáng tin hơn.


In [ ]:
import sys, time, unicodedata
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.builtin.mediapipe_builtin import LandmarkerLoader, VisionRunningMode
from src.builtin.mediapipe_builtin.defs import PoseLandmarkerPath, HandLandmarkerPath
from src.data.augment import augment_sample
from src.data.data_reader import DataHandler
from src.features.extract import extract, FEATURE_DIM, FEATURE_VERSION
from src.features.camera import open_camera
from src.features.frame import FRAME_SIZE, frame_padding

CLASS_PLAN = {}
cls = input("Nhap chu cai: ")
sample = int(input("So sample (gom ca mau quay): "))
if sample < 1:
    raise ValueError("So sample phai >= 1")
CLASS_PLAN[cls] = sample

NOISE_STRENGTH = 1.0
SEED = None

CAMERA_INDEX = 0
CAMERA_RESOLUTION = None
MIRROR_DATA = False
MIRROR_PREVIEW = True
COUNTDOWN_SECONDS = 1.0
RECORD_SECONDS = 2.0
MIN_POSE_COVERAGE = 0.8
DISPLAY_SIZE = 720
WINDOW = "VSL alphabet collect"

POSE_MODEL_PATH = Path(PoseLandmarkerPath)
HAND_MODEL_PATH = Path(HandLandmarkerPath)
DETECTION_CONF = dict(detection=0.5, presence=0.5, tracking=0.5)

SESSION = datetime.now().strftime("%Y%m%d-%H%M%S")
print("Session:", SESSION)

In [ ]:
handler = DataHandler()

assert FEATURE_DIM == DataHandler.FEATURE_DIM, (
    f"Extractor trả {FEATURE_DIM} chiều nhưng schema.FRAME_FEATURE_DIM = {DataHandler.FEATURE_DIM}. "
    "Cập nhật FRAME_FEATURE_DIM trong src/features/schema.py."
)
for p in (POSE_MODEL_PATH, HAND_MODEL_PATH):
    assert p.is_file(), f"Không thấy file model: {p}"

def print_table():
    counts = handler.get_class_counts()
    print(f"{'Lớp':<28}{'Số mẫu':>8}")
    print("-" * 36)
    for name, sample_count in counts.items():
        print(f"{name:<28}{sample_count:>8}")
    print("-" * 36)
    print(f"{len(counts)} lớp, {sum(counts.values())} mẫu")
    if handler.incomplete:
        print("\n[cảnh báo] mẫu dở dang/hỏng bị bỏ qua:")
        for path, reason in handler.incomplete:
            print("  ", path, "->", reason)

print_table()

In [ ]:
class Detector:
    """Pose + Hand landmarker (chế độ VIDEO, timestamp luôn tăng) -> đặc trưng 1 frame."""

    def __init__(self):
        c = DETECTION_CONF
        self.landmarkers = LandmarkerLoader(
            running_mode=VisionRunningMode.VIDEO,
            pose_model_path=POSE_MODEL_PATH, hand_model_path=HAND_MODEL_PATH,
            detection_confidence=c["detection"], presence_confidence=c["presence"],
            tracking_confidence=c["tracking"])

    def __call__(self, frame_bgr):
        pose_res, hand_res, _ = self.landmarkers.detect(frame_bgr)
        return extract(pose_res, hand_res), pose_res, hand_res

    def close(self):
        self.landmarkers.close()

def ascii_text(s):
    """cv2.putText không vẽ được chữ có dấu -> bỏ dấu khi hiển thị trên cửa sổ."""
    s = s.replace("đ", "d").replace("Đ", "D")
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()

def put(image, text, y, color=(255, 255, 255), scale=0.65):
    cv2.putText(image, text, (14, y), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(image, text, (14, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 2, cv2.LINE_AA)

POSE_DRAW = [0, 11, 12, 13, 14, 15, 16]

def draw_overlay(image, pose_res, hand_res):
    height, width = image.shape[:2]
    if pose_res and pose_res.pose_landmarks:
        for i in POSE_DRAW:
            landmark = pose_res.pose_landmarks[0][i]
            cv2.circle(image, (int(landmark.x * width), int(landmark.y * height)), 6, (0, 200, 255), -1)
    if hand_res and hand_res.hand_landmarks:
        for lms in hand_res.hand_landmarks:
            for landmark in lms:
                cv2.circle(image, (int(landmark.x * width), int(landmark.y * height)), 3, (0, 255, 0), -1)

def summarize(features):
    """(tỉ lệ frame có pose, tỉ lệ có tay trái, tỉ lệ có tay phải)"""
    stacked = np.stack(features)
    pres = np.nan_to_num(stacked[:, DataHandler.PRESENCE_COLS])
    return float((~np.isnan(stacked).all(axis=1)).mean()), float(pres[:, 0].mean()), float(pres[:, 1].mean())

def quality_ok(features, summary):
    return (len(features) >= DataHandler.MIN_FRAMES and summary[0] >= MIN_POSE_COVERAGE
            and (summary[1] > 0 or summary[2] > 0))

In [ ]:
def collect_class(capture, detector, class_name, n_samples, saved):
    """Thu thập `n_samples` mẫu MỚI cho 1 lớp; sample_idx đã lưu được thêm vào `saved`. Trả về "done" | "next" | "quit"."""
    name = handler.add_class(class_name)
    label = ascii_text(name)
    total = handler.get_class_counts().get(name, 0)
    state, phase_start = "READY", time.time()
    features, frames, summary, fps_est, message = [], [], None, 30.0, ""

    def log(text):
        nonlocal message
        message = ascii_text(text)
        print(text)

    while len(saved) < n_samples:
        has_frame, raw = capture.read()
        if not has_frame:
            raise RuntimeError("Không đọc được frame từ camera")
        if MIRROR_DATA:
            raw = cv2.flip(raw, 1)
        frame = frame_padding(raw)
        if frame.shape != (FRAME_SIZE, FRAME_SIZE, 3):
            raise RuntimeError(f"frame_padding phải trả ảnh {FRAME_SIZE}x{FRAME_SIZE}, nhận shape {frame.shape}")
        frame_features, pose_res, hand_res = detector(frame)
        now = time.time()

        if state == "COUNTDOWN" and now - phase_start >= COUNTDOWN_SECONDS:
            state, phase_start, features, frames = "RECORDING", now, [], []
        if state == "RECORDING":
            features.append(frame_features)
            frames.append(frame.copy())
            if now - phase_start >= RECORD_SECONDS:
                fps_est = len(features) / max(now - phase_start, 1e-6)
                summary = summarize(features)
                state = "REVIEW"

        display = cv2.resize(frame, (DISPLAY_SIZE, DISPLAY_SIZE), interpolation=cv2.INTER_AREA)
        draw_overlay(display, pose_res, hand_res)
        if MIRROR_PREVIEW:
            display = cv2.flip(display, 1)
        put(display, f"{label}  [{len(saved)}/{n_samples} moi | tong {total}]", 28, (255, 255, 0))
        if state == "READY":
            put(display, "SPACE: bat dau | U: xoa mau vua luu | N: lop ke | Q: thoat", 58)
        elif state == "COUNTDOWN":
            put(display, f"{max(0.0, COUNTDOWN_SECONDS - (now - phase_start)):.1f}", DISPLAY_SIZE // 2, (0, 200, 255), 3.0)
        elif state == "RECORDING":
            cv2.circle(display, (DISPLAY_SIZE - 30, 30), 12, (0, 0, 255), -1)
            put(display, f"DANG QUAY {now - phase_start:.1f}/{RECORD_SECONDS:.1f}s  ({len(features)} frame)", 58, (0, 0, 255))
        else:
            good = quality_ok(features, summary)
            put(display, f"{len(features)} frame | pose {summary[0]:.0%} | tay T {summary[1]:.0%} | tay P {summary[2]:.0%}",
                58, (0, 255, 0) if good else (0, 0, 255))
            put(display, "ENTER: luu | R: quay lai | F: luu du chat luong thap", 88)
        if message:
            put(display, message, DISPLAY_SIZE - 16, (0, 200, 255), 0.6)
        cv2.imshow(WINDOW, display)
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            return "quit"
        if key == ord("n"):
            return "next"
        if state == "READY":
            if key == 32:
                state, phase_start, message = "COUNTDOWN", now, ""
            elif key == ord("u") and saved:
                handler.remove_sample(name, saved.pop())
                total = handler.get_class_counts().get(name, 0)
                log("Đã xoá mẫu vừa lưu")
        elif state == "REVIEW":
            if key == ord("r"):
                state = "READY"
                log("Đã bỏ mẫu vừa quay")
            elif key in (13, 10, ord("f")):
                if key != ord("f") and not quality_ok(features, summary):
                    log("Chất lượng thấp: bấm F để vẫn lưu, hoặc R để quay lại")
                    continue
                try:
                    sample_index = handler.write_sample(
                        name, features, frames, fps=fps_est, session=SESSION,
                        meta={"mirror": MIRROR_DATA, "record_seconds": RECORD_SECONDS,
                              "feature_version": FEATURE_VERSION,
                              "pose_coverage": summary[0], "left_hand": summary[1], "right_hand": summary[2]})
                    saved.append(sample_index)
                    total = handler.get_class_counts().get(name, 0)
                    log(f"Đã lưu {name}/{sample_index} ({len(features)} frame, {fps_est:.1f} fps)")
                except ValueError as e:
                    log(f"Không lưu được: {e}")
                state = "READY"
    return "done"

## Chạy thu thập
Chạy ô dưới, bấm vào cửa sổ OpenCV để nhận phím. Mỗi lớp chỉ quay 1 mẫu; khi đóng camera, notebook tự tạo bản nhiễu.


In [ ]:
capture = open_camera(CAMERA_INDEX, CAMERA_RESOLUTION)
detector = Detector()
cv2.namedWindow(WINDOW, cv2.WINDOW_NORMAL)

recorded = {}
try:
    for class_name, sample_count in CLASS_PLAN.items():
        print(f"\n=== {class_name}: quay 1 mẫu, sau đó thêm {sample_count - 1} bản nhiễu ===")
        saved = []
        status = collect_class(capture, detector, class_name, 1, saved)
        if saved:
            recorded[class_name] = saved[-1]
        if status == "quit":
            break
finally:
    capture.release()
    detector.close()
    cv2.destroyAllWindows()
    for _ in range(4):
        cv2.waitKey(1)

rng = np.random.default_rng(SEED)
for class_name, sample_index in recorded.items():
    copies = CLASS_PLAN[class_name] - 1
    if copies < 1:
        continue
    source = handler._class_dir(handler.processed, class_name) / str(sample_index)
    created = augment_sample(source, handler.raw, copies, NOISE_STRENGTH, rng)
    print(f"{class_name}/{sample_index} -> {len(created)} bản nhiễu ({created[0].name}..{created[-1].name})")
handler.refresh_dataset()

print()
print_table()

## Kiểm tra & dọn dẹp

In [ ]:
rows = [row for row in handler.quality_report() if row["session"] == SESSION]
print(f"{len(rows)} mẫu trong session {SESSION}")
for row in sorted(rows, key=lambda row: row["pose_coverage"])[:10]:
    print(f"  {row['class']}/{row['sample_id']:<4} frames={row['frames']:<3} pose={row['pose_coverage']:.0%} "
          f"tayT={row['left_hand']:.0%} tayP={row['right_hand']:.0%}")